# Anime Colab Studio — ejecución supervisada en Google Colab

Selecciona el modelo, ComfyUI o AUTOMATIC1111, Cloudflare o ngrok y LoRAs. Las preferencias, modelos opcionales e imágenes PNG pueden conservarse en Google Drive.

**Uso recomendado:** ejecuta las celdas 1 y 2, elige tus preferencias, pulsa **Instalar / iniciar** y ejecuta la **celda 5 — Supervisar sesión activa**. La celda 5 seguirá ejecutándose mientras el servidor esté activo y te permitirá conocer su estado, incluso durante una generación.

**Importante:** la supervisión mantiene código real en ejecución, pero no impide que Google Colab cierre la VM al alcanzar límites de tiempo, saldo, GPU o condiciones del servicio. No simula clics ni evita restricciones del proveedor.

La celda 3 sirve para diagnóstico y la celda 4 permite instalar directamente desde una celda en ejecución, sin usar el botón. La celda 4 también supervisa el servidor al terminar la instalación.


In [ ]:
# @title 1. Instalar dependencias mínimas
%pip -q install -U ipywidgets huggingface_hub requests 'aiohttp>=3.10,<4'


In [ ]:
# @title 2. Abrir panel interactivo con progreso y logs recuperables
from google.colab import output
output.enable_custom_widget_manager()
from pathlib import Path
import importlib
import subprocess
import sys
project = Path("/content/ai_nsfw_source")
if not (project / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Mairon31/ai_nsfw.git",
                    str(project)], check=True)
else:
    subprocess.run(["git", "-C", str(project), "fetch", "--depth", "1",
                    "origin", "master"], check=True)
    subprocess.run(["git", "-C", str(project), "reset", "--hard",
                    "origin/master"], check=True)
sys.path.insert(0, str(project))
importlib.invalidate_caches()
for name in list(sys.modules):
    if name == "studio" or name.startswith("studio."):
        del sys.modules[name]
from studio.dashboard import make_dashboard
make_dashboard()


In [ ]:
# @title 3. Diagnóstico (ejecuta esta celda si el panel no avanza o quieres ver el progreso)
# Puedes repetir esta celda cuantas veces quieras mientras el instalador trabaja.
import json
from pathlib import Path
from studio import runtime
state = runtime.snapshot()
print("ESTADO:", state.get("stage", "sin iniciar"))
print("MENSAJE:", state.get("message", "sin datos"))
print("AVANCE ESTIMADO:", state.get("percent", 0), "%")
print("COMPROBACIÓN EXTERNA:", state.get("tunnel_verified", "pendiente"))
if state.get("access_password"):
    print("CONTRASEÑA DE SESIÓN:", state["access_password"])
elif state.get("password_source") == "WEBUI_PASSWORD":
    print("CONTRASEÑA: usa el secreto WEBUI_PASSWORD de Colab")
if state.get("url"):
    print("URL REMOTA:", state["url"])
print("PROCESOS:", json.dumps(runtime.read_json(runtime.PIDS), ensure_ascii=False))
print("ACTIVO:", runtime.process_active())
print("ÚLTIMOS LOGS (instalador + UI + túnel):")
print(runtime.recent_log(65))


### Inicio directo (alternativa)
Si el botón del panel se queda en *Preparando*, guarda preferencias, activa `INICIAR_EN_ESTA_CELDA` y ejecuta la siguiente celda. Verás el instalador en la consola activa. **No hace falta usar este modo si el botón funciona.**


In [ ]:
# @title 4. Opcional: iniciar en esta celda y ver cada log en vivo
# Activa esta casilla SOLO si al usar el botón "Instalar / iniciar" se atasca.
INICIAR_EN_ESTA_CELDA = False # @param {type:"boolean"}
if INICIAR_EN_ESTA_CELDA:
    from studio import launcher, runtime
    print("Modo directo de Colab: preferencias cargadas desde Google Drive.")
    cfg = launcher.load_settings()
    secrets = {k: launcher.secret(k) for k in
               ("HF_TOKEN", "NGROK_AUTHTOKEN", "CIVITAI_API_TOKEN", "WEBUI_PASSWORD")}
    codigo = runtime.launch_foreground(cfg, secrets)
    estado = runtime.snapshot()
    print("\n--- ESTADO FINAL ---")
    print(estado.get("stage"), ":", estado.get("message"))
    if estado.get("url"):
        print("ABRIR INTERFAZ:", estado["url"])
else:
    print("Modo directo desactivado; utiliza el panel de la celda 2.")


### 5. Supervisar sesión activa
Inicia el servidor desde el panel de la celda 2 y ejecuta esta celda. Puedes usar la Web UI mientras la celda muestra el estado real de la API. La celda permanecerá **ejecutándose** hasta que detengas el monitor o finalice el servidor. La supervisión no modifica los límites de Colab.

Si usas la celda 4 de inicio directo, **no necesitas ejecutar esta celda**, porque esa opción ya incluye supervisión continua.


In [ ]:
# @title 5. Mantener la celda ejecutándose mientras supervisa la Web UI
SUPERVISAR_SESION = True # @param {type:"boolean"}
INTERVALO_SEGUNDOS = 20 # @param {type:"slider", min:10, max:120, step:10}
if SUPERVISAR_SESION:
    from studio import runtime
    print("Consultando procesos de Anime Colab Studio...")
    try:
        resultado = await runtime.monitor_active_session_async(
            interval=INTERVALO_SEGUNDOS
        )
        print("Monitor finalizado con estado:", resultado)
    except KeyboardInterrupt:
        print("Supervisión interrumpida. El servidor puede continuar activo; detenlo desde el panel si quieres cerrarlo.")
else:
    print("Supervisión desactivada: el notebook funciona como antes.")
